Collecting from jolpica API

In [20]:
import requests
import pandas as pd
import time

BASE_URL = "https://api.jolpi.ca/ergast/f1/"

def get_race_results(season, retries=3):
    url = f"{BASE_URL}{season}/results/?limit=10000&format=json"

    for attempt in range(retries):
        response = requests.get(url)

        if response.status_code == 200:
            races = response.json()["MRData"]["RaceTable"]["Races"]
            race_data = []

            for race in races:
                for result in race["Results"]:
                    # Fastest lap time conversion
                    fastest_lap_time = None
                    if "FastestLap" in result and "Time" in result["FastestLap"]:
                        time_str = result["FastestLap"]["Time"]["time"]
                        mins, secs = map(float, time_str.split(":"))
                        fastest_lap_time = mins * 60 + secs

                    birth_year = int(result["Driver"]["dateOfBirth"][:4])
                    driver_age = season - birth_year

                    race_data.append({
                        "Season": season,
                        "Round": int(race["round"]),
                        "Race Name": race["raceName"],
                        "Date": race["date"],
                        "Circuit": race["Circuit"]["circuitName"],
                        "Country": race["Circuit"]["Location"]["country"],
                        "Latitude": float(race["Circuit"]["Location"]["lat"]),
                        "Longitude": float(race["Circuit"]["Location"]["long"]),
                        "Driver": f"{result['Driver']['givenName']} {result['Driver']['familyName']}",
                        "Driver Nationality": result["Driver"]["nationality"],
                        "Driver Age": driver_age,
                        "Constructor": result["Constructor"]["name"],
                        "Grid Position": int(result["grid"]),
                        "Race Position": int(result["position"]),
                        "Points Earned": float(result["points"]),
                        "Fastest Lap (s)": fastest_lap_time,
                        "Fastest Lap Rank": int(result["FastestLap"]["rank"]) if "FastestLap" in result else -1,
                        "Laps Completed": int(result["laps"]) if "laps" in result else -1,
                        "Status": result["status"],
                    })
            return race_data

        elif response.status_code == 429:
            wait_time = (attempt + 1) * 10
            print(f"  ⚠️ Rate limited for {season}. Waiting {wait_time}s before retry {attempt+1}/{retries}...")
            time.sleep(wait_time)

        else:
            print(f" Error for {season}: {response.status_code}")
            return []

    print(f" Failed to fetch {season} after {retries} retries — skipping.")
    return []

# Fetch 1950–2024
all_data = []
failed_years = []

for year in range(1950, 2024):
    print(f"Fetching {year}...")
    result = get_race_results(year)

    if result:
        all_data.extend(result)
        print(f"{year}: {len(result)} records fetched")
    else:
        failed_years.append(year)

    time.sleep(2)

df = pd.DataFrame(all_data)
print(f"\nTotal records: {len(df)}")
print(f" Columns: {df.shape[1]}")

if failed_years:
    print(f"\nThese years failed and need a re-run: {failed_years}")

print(df.head())

# Save locally
df.to_csv("F1_RaceIQ_Dataset_1950_2024.csv", index=False)
print("Saved as F1_RaceIQ_Dataset_1950_2024.csv")

# Save to Google Drive
from google.colab import drive
drive.mount('/content/drive')
df.to_csv('/content/drive/MyDrive/F1_RaceIQ_Dataset_1950_2024.csv', index=False)
print(" Saved to Google Drive")

Fetching 1950...
  ✅ 1950: 100 records fetched
Fetching 1951...
  ✅ 1951: 100 records fetched
Fetching 1952...
  ✅ 1952: 100 records fetched
Fetching 1953...
  ✅ 1953: 100 records fetched
Fetching 1954...
  ✅ 1954: 100 records fetched
Fetching 1955...
  ✅ 1955: 100 records fetched
Fetching 1956...
  ✅ 1956: 100 records fetched
Fetching 1957...
  ✅ 1957: 100 records fetched
Fetching 1958...
  ✅ 1958: 100 records fetched
Fetching 1959...
  ✅ 1959: 100 records fetched
Fetching 1960...
  ✅ 1960: 100 records fetched
Fetching 1961...
  ✅ 1961: 100 records fetched
Fetching 1962...
  ✅ 1962: 100 records fetched
Fetching 1963...
  ✅ 1963: 100 records fetched
Fetching 1964...
  ✅ 1964: 100 records fetched
Fetching 1965...
  ✅ 1965: 100 records fetched
Fetching 1966...
  ✅ 1966: 100 records fetched
Fetching 1967...
  ✅ 1967: 100 records fetched
Fetching 1968...
  ✅ 1968: 100 records fetched
Fetching 1969...
  ✅ 1969: 100 records fetched
Fetching 1970...
  ✅ 1970: 100 records fetched
Fetching 1971